## Documentacion oficial de PyTorch

- `torch.nn.RNN`: https://pytorch.org/docs/stable/generated/torch.nn.RNN.html

## Objetivo del notebook

La idea de este bloque es cambiar hiperparámetros y dimensiones de entrada para observar cómo cambian:
- los tensores de entrada y salida,
- el estado oculto final,
- los parámetros entrenables del modelo.

En todos los ejemplos vamos a usar la convencion de PyTorch con `batch_first=True`:

- `x`: `[batch_size, largo_secuencia, input_size]`
- `output`: `[batch_size, largo_secuencia, hidden_size]`
- `h_n`: `[num_layers, batch_size, hidden_size]`


## Funciones comunes a utilizar

In [ ]:
# Librerias
import torch


In [ ]:
# Clase minima para inspeccionar una RNN basica
class SimpleRNN(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=1, num_layers=1):
        super().__init__()
        self.rnn = torch.nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

    def forward(self, x):
        output, h_n = self.rnn(x)
        return output, h_n


In [ ]:
# Mostramos nombre y tamaño de cada parámetro entrenable.

def imp_param(model):
    print('-' * 84)
    print('PARAMETROS DEL MODELO')
    print('-' * 84)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f'{name}: {tuple(param.shape)}')
            total += param.numel()
    print()
    print(f'Total de parámetros entrenables: {total}')


In [ ]:
# Esta función imprime de forma ordenada la salida de un modelo.
#
# La usamos porque, según el modelo, la salida puede venir en distintos formatos:
# - un tensor único,
# - una tupla o lista de tensores,
# - o incluso un diccionario.
#
# En RNN simples de PyTorch, por ejemplo, el forward suele devolver:
#   output, h_n
# o sea, una tupla con dos tensores.
#
# La idea es recorrer recursivamente esa estructura y mostrar, para cada elemento,
# su nombre y su shape, sin asumir de antemano c?mo viene empaquetada la salida.
def mostrar_tensores(obj, nombre='salida'):
    # Caso 1: el objeto ya es un tensor. Mostramos su shape y su contenido.
    if isinstance(obj, torch.Tensor):
        print(f'{nombre}: shape = {tuple(obj.shape)}')
        print(obj)
    # Caso 2: el objeto es una tupla o lista. Recorremos cada posición.
    elif isinstance(obj, (tuple, list)):
        for i, item in enumerate(obj):
            mostrar_tensores(item, nombre=f'{nombre}[{i}]')
    # Caso 3: el objeto es un diccionario. Recorremos cada clave.
    elif isinstance(obj, dict):
        for key, value in obj.items():
            mostrar_tensores(value, nombre=f'{nombre}["{key}"]')
    # Caso 4: cualquier otro tipo. Lo mostramos para detectar salidas no esperadas.
    else:
        print(f'{nombre}: {type(obj)}')
        print(obj)


# Esta función arma una entrada aleatoria, ejecuta un forward y resume las shapes principales.
def teoria(model, largo_entrada=3, batch_size=1, input_size=1):
    print('-' * 84)
    print('MODELO')
    print('-' * 84)
    print(model)
    imp_param(model)

    entrada = torch.rand(batch_size, largo_entrada, input_size)
    print('-' * 84)
    print('ENTRADA')
    print('-' * 84)
    print(f'entrada shape: {tuple(entrada.shape)}')
    print(entrada)

    salida = model(entrada)
    print('-' * 84)
    print('SALIDA')
    print('-' * 84)
    mostrar_tensores(salida)

    return entrada, salida


## Ejercicio 1 - Resolución

Se mantienen exactamente los mismos tres ejemplos del notebook de teoria.


```
EJEMPLO A
input_size = 2
batch_size = 16
hidden_size = 24
num_layers = 3
```


In [ ]:
# Resolución del ejercicio 1, caso A.

input_size = 2
batch_size = 16
hidden_size = 24
num_layers = 3
largo_entrada = 5

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


```
EJEMPLO B
input_size = 6
batch_size = 16
hidden_size = 24
num_layers = 1
```


In [ ]:
# Resolución del ejercicio 1, caso B.

input_size = 6
batch_size = 16
hidden_size = 24
num_layers = 1
largo_entrada = 5

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


```
EJEMPLO C
input_size = 10
batch_size = 64
hidden_size = 64
num_layers = 4
```


In [ ]:
# Resolución del ejercicio 1, caso C.

input_size = 10
batch_size = 64
hidden_size = 64
num_layers = 4
largo_entrada = 3

model = SimpleRNN(input_size, hidden_size, num_layers)
teoria(model, largo_entrada=largo_entrada, batch_size=batch_size, input_size=input_size)


## Ejercicio 2 - Resolucion

La red devuelve `logits` de shape `[batch_size, n_clases]`. Si luego se quieren probabilidades, se aplica `softmax` fuera del `forward`.


In [ ]:
# Implementamos una RNN de clasificación que devuelve logits.

class RNNClasificacion(torch.nn.Module):
    def __init__(self, input_size=2, hidden_size=40, num_layers=2, n_clases=5):
        super().__init__()
        self.rnn = torch.nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = torch.nn.Linear(hidden_size, n_clases)

    def forward(self, x):
        output, h_n = self.rnn(x)
        ultimo_estado = output[:, -1, :]
        logits = self.fc(ultimo_estado)
        return logits


In [ ]:
# Probamos la arquitectura con una entrada aleatoria para verificar shapes de salida.

input_size = 2
hidden_size = 40
num_layers = 2
n_clases = 5
batch_size = 3
largo_entrada = 5

modelo_clas = RNNClasificacion(input_size, hidden_size, num_layers, n_clases)
entrada, logits = teoria(
    modelo_clas,
    largo_entrada=largo_entrada,
    batch_size=batch_size,
    input_size=input_size,
)


In [ ]:
# Aplicamos softmax fuera del modelo para interpretar probabilidades por clase.

probabilidades = torch.softmax(logits, dim=-1)
print('probabilidades shape =', tuple(probabilidades.shape))
print(probabilidades)
print('suma por muestra =', probabilidades.sum(dim=-1))
